# Full Agentic Pipeline Demo

Decision -> Human Review -> Feedback Storage -> RL/Supervised Learning demo for Colab.

In [ ]:
!pip install fastapi pydantic uvicorn pandas pyarrow scikit-learn torch transformers accelerate -q

In [ ]:
!git clone https://github.com/Lawapaul/AI_Agentic_DL.git

import os
import sys

repo_path = '/content/AI_Agentic_DL'
os.chdir(repo_path)
sys.path.insert(0, repo_path)

!git fetch origin
!git checkout codex/retraining-loop-system
!git pull origin codex/retraining-loop-system

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'
X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]
X = X.reshape(X.shape[0], -1)

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)
preds = model.predict(X)
probs = model.predict_proba(X)

In [ ]:
attack_map = {
    0: 'Normal Traffic',
    1: 'DoS Attack',
    2: 'Port Scan',
    3: 'Brute Force Attack',
    4: 'Web Attack',
    5: 'Botnet',
    6: 'Infiltration',
}

events = []
for idx in range(5):
    confidence = float(max(probs[idx]))
    attack_name = attack_map.get(int(preds[idx]), 'Unknown Attack')
    suggested_action = 'No Action' if attack_name == 'Normal Traffic' else 'Block' if confidence > 0.85 else 'Alert' if confidence > 0.65 else 'Monitor'
    events.append({
        'event_id': f'event-{idx}',
        'decision_output': {
            'event_id': f'event-{idx}',
            'risk_score': confidence,
            'uncertainty': float(1.0 - confidence),
            'explanation': f'{attack_name} detected from CICIDS-derived test sample.',
            'suggested_action': suggested_action,
        },
    })

events[0]

In [ ]:
from feedback_store.feedback_db import fetch_feedback, init_db
from integration.pipeline_connector import AgenticPipelineConnector

results_dir = os.path.join(repo_path, 'experiments', 'results')
db_path = os.path.join(results_dir, 'feedback.db')
init_db(db_path)

connector = AgenticPipelineConnector(
    db_path=db_path,
    results_dir=results_dir,
    risk_threshold=0.7,
    uncertainty_threshold=0.25,
)

In [ ]:
run_outputs = []
for event in events:
    result = connector.run_full_pipeline(event)
    run_outputs.append(result)
    print('\nEvent:', result['decision_output']['event_id'])
    print('Decision:', result['decision_output'])
    print('Human Review:', result['review_output'])
    print('Feedback:', result['feedback_record'])

In [ ]:
feedback_rows = fetch_feedback(db_path)
print('Stored feedback rows:', len(feedback_rows))
feedback_rows[:2]

In [ ]:
print('Updated RL policy:')
connector.rl_trainer.q_table